# Neuro-5P pilot v2: synthetic data, redesigned conflict signal

**Question.** Do five specialist monitors that watch for *inconsistency between related signals* catch failures that (B) one tuned combined score or (C) a plain "any monitor fails" rule miss, at the same false alarm rate?

**Why v1 failed.** In v1, "conflict = one monitor passes, one fails" always contains a failure, so rule C had already flagged it. D could never beat C.

**The fix (this notebook).** In real systems some signals normally move together. Here the model leans on a proxy feature, so the parity gap (P2) rises and falls with the proxy correlation (P3). A *conflict* is now: **P2 and P3 stop agreeing with each other**, even when each one alone still looks normal. That is information a "flag if any single monitor fails" rule cannot see.

Conditions (all see the same 5 signals, all compared at the **same false positive rate**):
- **B** one combined score, weights tuned on labelled faults (signed weights, two-sided)
- **C** five monitors, flag if any one is abnormal
- **D** the same five monitors plus a coordinator that watches the P2-P3 consistency

Condition A (no monitor) is just the ground truth: we know when each fault was injected. Run the cells top to bottom (about 2 to 3 minutes). Needs only numpy and matplotlib.

## Cell 1: settings
Stream length `N`, rolling window `W` for every monitor, and where each fault starts and ends. Faults are spaced at least `W` apart, so one fault's tail never leaks into the next fault's window.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

N, W = 2800, 200
VIOL = {"V5": (300, 550), "V3": (800, 1050), "V1": (1300, 1550), "V2": (1800, 2050), "V4": (2300, 2550)}
NAMES = {"V1": "accuracy drift", "V2": "parity w/o proxy", "V3": "masked proxy",
         "V4": "process gap", "V5": "participation gap"}
CONFLICT = ["V2", "V3"]    # the two faults designed to break the P2-P3 relationship

## Cell 2: simulate the model's outputs
A hidden number `s` (proxy strength) drifts slowly over time. It sets how strongly the `city` feature is tied to `group`. The model copies `city` half the time, so **when `s` is high the parity gap is high too**. This is the normal coupling between P2 and P3.

Faults, each lasting 250 inferences:
- **V1 accuracy drift:** accuracy drops from 85% to 65% (single-monitor fault)
- **V2 parity without proxy:** group 1 loses some positive predictions while the proxy is weaker. P2 rises but P3 does not
- **V3 masked proxy:** city-group tie gets stronger but the model stops using it. P3 rises but P2 does not
- **V4 process gap:** human review rate falls from 95% to 30% (single-monitor fault)
- **V5 participation gap:** 30% of outputs come from an unknown group (single-monitor fault)

`mag` scales V2 and V3. V1, V4 and V5 are the controls where we expect **no** advantage for D.

In [ ]:
def make_stream(seed, inject=True, mag=1.0):
    rng = np.random.default_rng(seed)
    s = np.zeros(N); s[0] = 0.3                                  # proxy strength, slow random walk
    for i in range(1, N):
        s[i] = np.clip(s[i-1] + 0.02*(0.3 - s[i-1]) + 0.03*rng.normal(), 0, 0.7)
    use = np.full(N, 0.5)                                        # how often the model copies city
    acc = np.full(N, 0.85); rev = np.full(N, 0.95)
    unk = np.full(N, 0.02); extra_flip = np.zeros(N)
    if inject:
        a, b = VIOL["V3"]; s[a:b] = np.clip(s[a:b] + 0.25*mag, 0, 0.9); use[a:b] = 0.0
        a, b = VIOL["V1"]; acc[a:b] = 0.65
        a, b = VIOL["V2"]; s[a:b] = np.clip(s[a:b] - 0.15*mag, 0, 1); extra_flip[a:b] = 0.25*mag
        a, b = VIOL["V4"]; rev[a:b] = 0.30
        a, b = VIOL["V5"]; unk[a:b] = 0.30
    g = rng.integers(0, 2, N); y = rng.integers(0, 2, N)
    city = np.where(rng.random(N) < s, g, rng.integers(0, 2, N))
    p = np.where(rng.random(N) < acc, y, 1 - y)
    p = np.where(rng.random(N) < use, city, p)
    p = np.where((g == 1) & (p == 1) & (rng.random(N) < extra_flip), 0, p)
    return dict(g=g, y=y, city=city, p=p, rev=rng.random(N) < rev, known=rng.random(N) >= unk)

## Cell 3: the five monitors
Each looks at the last `W` outputs and returns one number:
P1 error rate, P2 parity gap (group 1 minus group 0), P3 correlation of city with group, P4 share unreviewed, P5 share from unknown group.

In [ ]:
def signals(d):
    X = np.full((N, 5), np.nan)
    for t in range(W - 1, N):
        a = slice(t - W + 1, t + 1); g, p, y = d["g"][a], d["p"][a], d["y"][a]
        X[t, 0] = 1 - (p == y).mean()
        X[t, 1] = p[g == 1].mean() - p[g == 0].mean()
        X[t, 2] = np.corrcoef(d["city"][a], g)[0, 1]
        X[t, 3] = 1 - d["rev"][a].mean()
        X[t, 4] = 1 - d["known"][a].mean()
    return X[W - 1:]

t = np.arange(W - 1, N)            # inference index of each signal row

## Cell 4: learn "normal" from fault-free streams only
We simulate 60 **clean** streams and record the mean and spread of each signal. We also check the key assumption: **P2 and P3 should be correlated in clean data.** If they were not, the conflict signal would carry no extra information and D could not beat C.

`beta` is how much P2 normally moves per unit of P3, and `res_sd` is the normal spread of what is left over. The conflict score later measures how far P2 is from what P3 predicts.

In [ ]:
clean = np.vstack([signals(make_stream(1000 + i, inject=False)) for i in range(60)])
mu, sd = clean.mean(0), clean.std(0) + 1e-9
z = lambda X: (X - mu) / sd
zc = z(clean)
beta = (zc[:, 1] @ zc[:, 2]) / (zc[:, 2] @ zc[:, 2])
res_sd = (zc[:, 1] - beta * zc[:, 2]).std()
print("Clean correlation between P2 and P3:", round(np.corrcoef(zc[:, 1], zc[:, 2])[0, 1], 2))
print("beta =", round(beta, 2), "  residual sd =", round(res_sd, 2))

## Cell 5: one score per condition (higher = more abnormal)
- **C:** the largest standardised signal across the five monitors, each counted in either direction (an unexpected improvement is also a change worth flagging)
- **D:** the same, plus the consistency score `|P2 - beta*P3| / res_sd`, which is large when P2 and P3 disagree in *either* direction
- **B:** `|w . z|`, a single weighted score. The weights are found by trying 2,000 random **signed** weightings on a separate stream that **contains labelled faults**, keeping the one with the best recall at a 5% false positive rate. This is the strongest simple version of the combined-score baseline that Dr Roy asked for.

No thresholds are chosen here. Every condition is compared at the same false positive rate in the next cell.

In [ ]:
def comps(X):                                    # standardised per-monitor scores for C
    return np.abs(z(X))

def conflict(X):                                 # P2 and P3 disagree (either direction)
    zz = z(X); return np.abs(zz[:, 1] - beta * zz[:, 2]) / res_sd

sC = lambda X: comps(X).max(1)
sD = lambda X: np.maximum(sC(X), conflict(X))

def fault_mask():
    m = np.zeros(len(t), bool)
    for a, b in VIOL.values(): m |= (t >= a) & (t < b)
    return m

tune = signals(make_stream(2000)); tv = fault_mask()
rng = np.random.default_rng(0)
Wm = np.vstack([np.ones(5) / 5, rng.normal(size=(2000, 5))])
Wm /= np.abs(Wm).sum(1, keepdims=True)
thr = np.quantile(np.abs(zc[::10] @ Wm.T), 0.95, axis=0)
rec = (np.abs(z(tune)[tv] @ Wm.T) > thr).mean(0)
wB = Wm[rec.argmax()]
sB = lambda X: np.abs(z(X) @ wB)
print("Condition B weights (P1..P5):", np.round(wB, 2))
SCORE = {"B": sB, "C": sC, "D": sD}

## Cell 6: test streams and the false positive reference
- 20 **fault-free** held-out streams give the distribution of scores when nothing is wrong. The threshold for a given false positive rate comes from here.
- 20 new streams **with** all five faults give the recall inside each fault window.

In [ ]:
held = [signals(make_stream(5000 + i, inject=False)) for i in range(20)]
tests = {1.0: [signals(make_stream(3000 + i, True, 1.0)) for i in range(20)]}
held_scores = {c: np.concatenate([f(X) for X in held]) for c, f in SCORE.items()}

def recall_at(fpr, Xs, conds="BCD"):
    out = {c: {v: [] for v in VIOL} for c in conds}
    for c in conds:
        thr = np.quantile(held_scores[c], 1 - fpr)
        for X in Xs:
            s = SCORE[c](X)
            for v, (a, b) in VIOL.items():
                out[c][v].append((s[(t >= a) & (t < b)] > thr).mean())
    return {c: {v: np.array(x) for v, x in d.items()} for c, d in out.items()}

R = recall_at(0.05, tests[1.0])

## Cell 7: results at a 5% false positive rate
Recall is the share of inferences inside a fault window that are flagged. Mean over 20 streams. V2 and V3 are the conflict faults, the rest are controls where D should only tie C.

In [ ]:
print(f"{'':3}" + "".join(f"{v} {NAMES[v][:11]}".rjust(18) for v in VIOL))
for c in "BCD":
    print(f"{c:3}" + "".join(f"{R[c][v].mean():18.2f}" for v in VIOL))

def boot_ci(x, n=2000, seed=0):
    r = np.random.default_rng(seed); m = [r.choice(x, len(x)).mean() for _ in range(n)]
    return np.percentile(m, [2.5, 97.5])

print()
print("Paired difference in recall, 95% bootstrap CI over the 20 streams")
for ref in "CB":
    for v in VIOL:
        d = R["D"][v] - R[ref][v]; lo, hi = boot_ci(d)
        print(f"  D - {ref}  {v} {NAMES[v]:18s} {d.mean():+.2f}  [{lo:+.2f}, {hi:+.2f}]")

## Cell 8: recall vs false positive rate (the trade-off curve)
Pooled over the two conflict faults. A curve higher and to the left is better. This is the figure Dr Roy asked for.

In [ ]:
grid = [0.005, 0.01, 0.02, 0.03, 0.05, 0.08, 0.12, 0.2]
curve = {c: [] for c in "BCD"}
for f in grid:
    Rf = recall_at(f, tests[1.0])
    for c in "BCD":
        curve[c].append(np.mean([Rf[c][v].mean() for v in CONFLICT]))

fig, ax = plt.subplots(figsize=(6, 4))
for c, lab in zip("BCD", ["B combined score", "C any monitor fails", "D + consistency check"]):
    ax.plot(grid, curve[c], marker="o", label=lab)
ax.set_xlabel("false positive rate"); ax.set_ylabel("recall on conflict faults (V2, V3)")
ax.set_ylim(0, 1.02); ax.legend(frameon=False); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

## Cell 9: recall at 5% FPR for every fault
Bars for B, C, D. V1, V4 and V5 are the control faults.

In [ ]:
x = np.arange(len(VIOL)); w = 0.27
fig, ax = plt.subplots(figsize=(8, 4))
for k, c in enumerate("BCD"):
    ax.bar(x + (k - 1) * w, [R[c][v].mean() for v in VIOL], w,
           yerr=[R[c][v].std() for v in VIOL], capsize=2, label={"B": "B", "C": "C", "D": "D"}[c])
ax.set_xticks(x); ax.set_xticklabels([f"{v}\n{NAMES[v]}" for v in VIOL], fontsize=8)
ax.set_ylabel("recall at 5% FPR"); ax.set_ylim(0, 1.05); ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

## Cell 10: does the result depend on how big we made the conflict faults?
A reviewer will say we engineered faults that D happens to catch. So we shrink and grow the strength of V2 and V3 (`mag`) and see where D's advantage appears and disappears. If D only wins at one hand-picked size, that is a weak result.

In [ ]:
for m in (0.5, 1.5):
    tests[m] = [signals(make_stream(3000 + i, True, m)) for i in range(20)]
print("Recall at 5% FPR, mean of V2 and V3")
print(f"{'mag':>5}{'B':>8}{'C':>8}{'D':>8}")
for m in (0.5, 1.0, 1.5):
    Rm = recall_at(0.05, tests[m])
    print(f"{m:5.1f}" + "".join(f"{np.mean([Rm[c][v].mean() for v in CONFLICT]):8.2f}" for c in "BCD"))

## How to read this, and what it does not show
**Corrected after a validation pass.** An earlier version of this notebook reported D beating C by about +0.34 on V3. That was an artefact: condition C only watched P1 (accuracy) for increases, but in V3 the model stops copying `city`, so accuracy *improves* by about 3 standard deviations. A one-sided C missed it. With every monitor counted in either direction (the fair version, used above), the gap shrinks to about +0.06.

**Numbers from the reference run** (seeded; yours should match closely, not necessarily to the last digit). Recall at 5% false positive rate:

| | V3 masked proxy | V2 parity w/o proxy | Controls V1 / V4 / V5 |
|---|---|---|---|
| B combined score | 0.59 | 0.06 | 0.26 / 0.86 / 0.87 |
| C any monitor abnormal | 0.61 | 0.21 | 0.36 / 0.95 / 0.92 |
| D + consistency check | 0.67 | 0.22 | 0.36 / 0.95 / 0.92 |

**Honest reading**
- D beats C only on V3, by about +0.06 (95% CI +0.02 to +0.09). On V2 the difference is +0.01 (CI -0.02 to +0.04), which is nothing.
- A multivariate baseline (Mahalanobis distance over all five signals, learned from clean data only) scored 0.69 on V3 and 0.19 on V2 in a separate check, so it matches D. The claim "invisible to any single score" holds only for weighted sums of per-dimension scores.
- D's clearest edge is over B (the weighted sum), which has to compromise one set of weights across all five faults.

**Validation checks run on this design (what they found)**
1. **Faults are not clean.** V3 also moves accuracy by about 3 sd; V2 also moves P3. Fix the generator so a fault changes only what it claims to change (for example, hold overall accuracy constant in V3).
2. **The "control" faults are too easy.** V4 and V5 shift their signals by 16 to 26 sd, so every method hits the ceiling and a tie proves little. Use controls near the detection limit.
3. **False positive rates are close but not exact.** On a third fault-free set at a 5% target: B 0.039, C 0.043, D 0.053, M5 0.045. Per-stream FPR varies by about 0.04 because scores are strongly autocorrelated (lag-1 correlation 0.98).
4. **Placebo test.** On fault-free streams, D scores about +0.01 above C by chance of calibration alone, so any D-C gap under about 0.02 is not evidence.
5. **Power.** The V2 difference would need roughly 40 or more streams to detect reliably; 20 is not enough.
6. **No leakage.** Calibration, B tuning, tests and FPR reference sets use disjoint seeds.

**Next steps:** fix fault specificity, add near-threshold controls, add the multivariate baseline and a coupling sweep (no coupling should mean no D advantage), use 50 or more streams, then repeat on real data.